# Friend A — resumable dense fine-tune pipeline (revised 2026-09-10)

This single root notebook follows the current `TEAM.md` / `PROMPT_FRIEND_A.md`. Dense pseudo-labels are used to
**fine-tune the pilkwang primary**, not to train a model from scratch. It never submits to the leaderboard.

`Run All` performs:

1. finishes/reuses STEP 1c (`teacher_geff/`, `train_dense/`, `train_dense_geff/`);
2. runs the same-machine sparse primary fine-tune reference (1200 iterations);
3. runs dense primary variants: 1200, 2400, 1200 with `det-neg-weight=0.1`, and 1200 with intensity augmentation;
4. scores every run on the fixed 12-video `44b6` LOEO set with the official metric;
5. if the best dense variant beats the sparse reference by at least 0.005, extends dense labels to all 199 videos,
   fine-tunes the winner on the exp03 183-video split, uploads `friend_a_ft_dense_primary/`, and prints a hand-off message.

Every training command contains `--init-weights` pointing to the verified pilkwang primary checkpoint. The notebook
contains no from-scratch 20/30-epoch or all-199 scratch branch.

Expected time on the RTX 5070 Ti is about 7 hours through STEP 2 (the 2400-iteration run takes roughly twice as long
as a 1200-iteration run). If a dense recipe wins, STEP 3 adds roughly 2–3 hours including the missing teacher graphs,
dense build, two validations, and upload. These are estimates; the notebook prints and saves every subprocess log.

## Stop and resume

- Teacher inference is saved in 6-video chunks and dense GEFFs are saved per video.
- Each completed fine-tune variant and official evaluation is recorded and skipped on rerun.
- If interrupted inside a roughly one-hour variant, only that active variant restarts; earlier variants remain complete.
- Keep `outputs/local_friend_a_exp03_loeo_dense_20ep/active/` (the existing 138 teacher files),
  `outputs/local_friend_a_exp03_dense_ft_primary/active/`, and `train_dense/`.

Start Jupyter from `C:\Projects\CellTracker`, select the project Python 3.12 kernel, and use **Run All**. For a dry
preflight with no GPU work, uploads, or commits:

```powershell
$env:BIOHUB_DENSE_FT_RUN_PIPELINE = "0"
jupyter lab
```


In [ ]:
from __future__ import annotations

import csv
import hashlib
import importlib.util
import json
import math
import os
import platform
import re
import shutil
import subprocess
import sys
import time
from datetime import datetime
from pathlib import Path


EXPERIMENT = "local_friend_a_exp03_dense_ft_primary"
LEGACY_STEP1C_EXPERIMENT = "local_friend_a_exp03_loeo_dense_20ep"
EXPECTED_BRANCH = "detector-baseline-v2"
WEIGHTS_DATASET = "vitaliineibauer/biohub-ft-weights"
TEACHER_DATASET_FOLDER = "teacher_geff"
DENSE_DATASET_FOLDER = "train_dense_geff"
WINNER_WEIGHTS_FOLDER = "friend_a_ft_dense_primary"

BATCH_SIZE = 4
NUM_WORKERS = 0 if os.name == "nt" else 4
FT_EPOCHS = 1
FT_LEARNING_RATE = 5e-5
SEED = 42
WINDOW_SIZE = 2
DOWNSAMPLE = "1,4,4"
POOL_KERNEL_UM = 5.0
DET_LOSS_WEIGHT = 1.0
MAX_DETECTIONS = 0
DET_THRESHOLD = 0.965
ILP_APPEARANCE = 0.0
ILP_DISAPPEARANCE = 2.0
ILP_DIVISION = 1.2
TEACHER_CHUNK_SIZE = 6
TEACHER_DEDUP_RADIUS_UM = 5.0
WINNING_DELTA = 0.005

VARIANTS = [
    {"name": "sparse_1200", "data": "sparse", "max_iters": 1200, "det_neg_weight": 0.01, "aug_intensity": False},
    {"name": "dense_1200", "data": "dense", "max_iters": 1200, "det_neg_weight": 0.01, "aug_intensity": False},
    {"name": "dense_2400", "data": "dense", "max_iters": 2400, "det_neg_weight": 0.01, "aug_intensity": False},
    {"name": "dense_1200_neg0p1", "data": "dense", "max_iters": 1200, "det_neg_weight": 0.1, "aug_intensity": False},
    {"name": "dense_1200_aug", "data": "dense", "max_iters": 1200, "det_neg_weight": 0.01, "aug_intensity": True},
]

RUN_PIPELINE = os.environ.get("BIOHUB_DENSE_FT_RUN_PIPELINE", "1") != "0"
AUTO_UPLOAD = os.environ.get("BIOHUB_DENSE_FT_AUTO_UPLOAD", "1") != "0"
AUTO_GIT = os.environ.get("BIOHUB_DENSE_FT_AUTO_GIT", "1") != "0"
RUN_STEP3 = os.environ.get("BIOHUB_DENSE_FT_RUN_STEP3", "1") != "0"
RUN_ID = os.environ.get("BIOHUB_DENSE_FT_RUN_ID", "active").strip() or "active"


def find_project_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        for repo_candidate in (candidate, candidate / ".github-push-worktree"):
            if (repo_candidate / "TEAM.md").exists() and (repo_candidate / "tools" / "repo" / "scripts").exists():
                return repo_candidate
    raise FileNotFoundError("Repository root not found. Start Jupyter inside C:\\Projects\\CellTracker.")


PROJECT_ROOT = find_project_root(Path.cwd())
WORKSPACE_ROOT = PROJECT_ROOT.parent if PROJECT_ROOT.name == ".github-push-worktree" else PROJECT_ROOT
GIT = ["git", "-c", f"safe.directory={PROJECT_ROOT.as_posix()}"]
DATA_DIR = Path(os.environ.get("BIOHUB_DATA_DIR", WORKSPACE_ROOT / "train")).expanduser().resolve()
DENSE_DATA_DIR = Path(os.environ.get("BIOHUB_DENSE_DATA_DIR", WORKSPACE_ROOT / "train_dense")).expanduser().resolve()
OUTPUT_ROOT = Path(os.environ.get("BIOHUB_OUTPUT_DIR", WORKSPACE_ROOT / "outputs")).expanduser().resolve()
RUN_DIR = OUTPUT_ROOT / EXPERIMENT / RUN_ID
LEGACY_STEP1C_RUN_DIR = OUTPUT_ROOT / LEGACY_STEP1C_EXPERIMENT / "active"
PIPELINE_STATE_PATH = RUN_DIR / "pipeline_state.json"
REPO_DIR = RUN_DIR / "repo"
PREDICTIONS_ROOT = RUN_DIR / "predictions"
EVALUATION_ROOT = RUN_DIR / "evaluation"
TEACHER_DIR = (
    LEGACY_STEP1C_RUN_DIR / TEACHER_DATASET_FOLDER
    if (LEGACY_STEP1C_RUN_DIR / TEACHER_DATASET_FOLDER).exists()
    else RUN_DIR / TEACHER_DATASET_FOLDER
)
MANIFEST_PATH = PROJECT_ROOT / "experiments" / "_lib" / "loeo_train6bba_val44b6.json"
EXP03_CONFIG_PATH = PROJECT_ROOT / "experiments" / "exp03_finetune_ab" / "config.py"
TRAIN_PATCH_PATH = PROJECT_ROOT / "experiments" / "_lib" / "train_patches.py"
PREDICT_PATCH_PATH = PROJECT_ROOT / "experiments" / "_lib" / "predict_patches.py"
BUILDER_PATH = PROJECT_ROOT / "tools" / "build_dense_gt.py"
EXPERIMENT_DIR = PROJECT_ROOT / "experiments" / EXPERIMENT
REFERENCE_ROOT = OUTPUT_ROOT / "_dense_ft_reference_weights"
PRIMARY_WEIGHTS_DIR = REFERENCE_ROOT / "pilkwang_primary"
PRIMARY_WEIGHTS = PRIMARY_WEIGHTS_DIR / "edge_predictor_best.pth"
PRIMARY_EXPECTED_SHA256 = "12f6881ee3620a831697ca098ff8f48e687a24225f4e048b538deec3562fe771"
PRIMARY_DATASET = "pilkwang/biohub-tracking-support-pack-50ep-v1"
PRIMARY_REMOTE_FOLDER = "weights/unet_transformer/split_0"
TEACHER_PRIMARY_DIR = REFERENCE_ROOT / "teacher_exp03B"
TEACHER_PRIMARY_WEIGHTS = TEACHER_PRIMARY_DIR / "edge_predictor_best.pth"
TEACHER_PRIMARY_EXPECTED_SHA256 = "f3bb1932efe8fbb6893f7707eebb0355814d53f86081e0d8c48d90140e98b305"
TEACHER_SECONDARY_DIR = REFERENCE_ROOT / "teacher_seed314159"
TEACHER_SECONDARY_WEIGHTS = TEACHER_SECONDARY_DIR / "edge_predictor_best.pth"
TEACHER_SECONDARY_EXPECTED_SHA256 = "9bac2fa0dadc4a6fc1899e0caf187f4b553e0a7cd90ba1261a68b35ffe9e305f"
SECONDARY_DATASET = "pilkwang/biohub-temporal-unet3d-seed314159-v1"
SECONDARY_REMOTE_FOLDER = "weights/unet_transformer/split_0"
ALL_MANIFEST_PATH = RUN_DIR / "all199.json"
LOEO_TRAIN_SPLIT_PATH = RUN_DIR / "loeo_train.json"
LOEO_EVAL_SPLIT_PATH = RUN_DIR / "loeo_eval.json"
EXP03_TRAIN_SPLIT_PATH = RUN_DIR / "exp03_train.json"
EXP03_EVAL_SPLIT_PATH = RUN_DIR / "exp03_eval.json"


def atomic_write_text(path: Path, value: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(value, encoding="utf-8")
    temporary.replace(path)


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def load_state() -> dict:
    if not PIPELINE_STATE_PATH.is_file():
        return {"experiment": EXPERIMENT, "run_id": RUN_ID, "status": "new", "variants": {}}
    return json.loads(PIPELINE_STATE_PATH.read_text(encoding="utf-8"))


def save_state(**updates) -> dict:
    state = load_state()
    state.update(updates)
    state["updated_at"] = datetime.now().astimezone().isoformat()
    atomic_write_text(PIPELINE_STATE_PATH, json.dumps(state, ensure_ascii=False, indent=2))
    return state


def terminate_process(process: subprocess.Popen) -> None:
    if process.poll() is not None:
        return
    process.terminate()
    try:
        process.wait(timeout=30)
    except subprocess.TimeoutExpired:
        process.kill()
        process.wait(timeout=30)


def stream_command(command: list[str], *, cwd: Path, env: dict, log_path: Path, label: str) -> None:
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print(f"\n[{label}] {subprocess.list2cmdline(command)}", flush=True)
    with log_path.open("a", encoding="utf-8") as log_handle:
        log_handle.write(f"\n===== {datetime.now().astimezone().isoformat()} | {label} =====\n")
        log_handle.flush()
        process = subprocess.Popen(
            command, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding="utf-8", errors="replace", bufsize=1,
        )
        try:
            assert process.stdout is not None
            for line in process.stdout:
                log_handle.write(line)
                log_handle.flush()
                encoding = getattr(sys.stdout, "encoding", None) or "utf-8"
                print(line.encode(encoding, errors="replace").decode(encoding, errors="replace"), end="", flush=True)
            return_code = process.wait()
        except KeyboardInterrupt:
            terminate_process(process)
            save_state(status=f"interrupted_during_{label}")
            print("\nStopped safely. Run All again; completed variants will be skipped.", flush=True)
            raise
        except BaseException:
            terminate_process(process)
            save_state(status=f"notebook_error_during_{label}")
            raise
    if return_code != 0:
        save_state(status=f"failed_during_{label}", failed_log=str(log_path), return_code=return_code)
        raise RuntimeError(f"{label} failed with exit code {return_code}; see {log_path}")


def checked(command: list[str], *, cwd: Path, env: dict | None = None) -> str:
    result = subprocess.run(
        command, cwd=cwd, env=env, text=True, encoding="utf-8", errors="replace", capture_output=True,
    )
    if result.returncode != 0:
        raise RuntimeError(
            f"Command failed ({result.returncode}): {subprocess.list2cmdline(command)}\n"
            + (result.stdout + result.stderr)[-6000:]
        )
    return result.stdout


def import_file(module_name: str, path: Path):
    spec = importlib.util.spec_from_file_location(module_name, path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f"Cannot import {path}")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


def common_env(data_dir: Path) -> dict[str, str]:
    env = {
        **os.environ,
        "BIOHUB_DATA_DIR": str(data_dir),
        "BIOHUB_PREDICTIONS_DIR": str(PREDICTIONS_ROOT),
        "PYTHONPATH": str(REPO_DIR / "src"),
        "PYTHONUNBUFFERED": "1",
        "PYTHONUTF8": "1",
        "PYTHONIOENCODING": "utf-8",
        "PYTORCH_ALLOC_CONF": "expandable_segments:True",
    }
    env.pop("BIOHUB_ENSEMBLE_WEIGHTS", None)
    return env


def guarded_remove_tree(path: Path, parent: Path) -> None:
    if not path.resolve().is_relative_to(parent.resolve()):
        raise RuntimeError(f"Refusing to remove path outside {parent}: {path}")
    if path.exists():
        shutil.rmtree(path)


def update_marked_block(path: Path, marker: str, line: str, heading_prefix: str) -> None:
    text_value = path.read_text(encoding="utf-8")
    block = marker + "\n" + line
    pattern = re.compile(re.escape(marker) + r"\n[^\n]*")
    if pattern.search(text_value):
        text_value = pattern.sub(block, text_value, count=1)
    else:
        heading = next((item for item in text_value.splitlines() if item.startswith(heading_prefix)), None)
        if heading is None:
            raise RuntimeError(f"Heading not found in {path}: {heading_prefix}")
        text_value = text_value.replace(heading, heading + "\n" + block, 1)
    atomic_write_text(path, text_value.rstrip() + "\n")


print(f"project:      {PROJECT_ROOT}")
print(f"sparse data:  {DATA_DIR}")
print(f"dense data:   {DENSE_DATA_DIR}")
print(f"teacher data: {TEACHER_DIR}")
print(f"run:          {RUN_DIR}")


In [ ]:
# Validate both fixed splits, prepare the patched repo, and resolve verified checkpoints.

import ast

required_modules = [
    "torch", "tracksdata", "geff", "polars", "rustworkx", "zarr", "tqdm", "scipy",
    "skimage", "dask", "networkx", "pandas", "pyscipopt", "ilpy", "blosc2", "kaggle",
]
missing_modules = [name for name in required_modules if importlib.util.find_spec(name) is None]
if missing_modules:
    raise ImportError("Missing pipeline packages: " + ", ".join(missing_modules))

import torch

if sys.version_info[:2] != (3, 12):
    raise RuntimeError(f"Python 3.12 is required; current interpreter: {sys.version.split()[0]}")
if not DATA_DIR.is_dir():
    raise FileNotFoundError(DATA_DIR)

loeo_manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
train_ids = list(loeo_manifest["train"])
holdout_ids = list(loeo_manifest["holdout"])
if len(train_ids) != 126 or len(holdout_ids) != 12 or set(train_ids) & set(holdout_ids):
    raise RuntimeError("Unexpected or leaking LOEO manifest")
if any(not value.startswith("6bba_") for value in train_ids):
    raise RuntimeError("LOEO train side must contain only 6bba")
if any(not value.startswith("44b6_") for value in holdout_ids):
    raise RuntimeError("LOEO validation side must contain only 44b6")


def read_exp03_split(path: Path) -> dict[str, list[str]]:
    tree = ast.parse(path.read_text(encoding="utf-8"), filename=str(path))
    for node in ast.walk(tree):
        if not isinstance(node, ast.Assign) or not isinstance(node.value, ast.Call):
            continue
        call = node.value
        if (
            isinstance(call.func, ast.Attribute)
            and isinstance(call.func.value, ast.Name)
            and call.func.value.id == "json"
            and call.func.attr == "dumps"
            and call.args
        ):
            value = ast.literal_eval(call.args[0])
            if isinstance(value, dict) and set(value) == {"train", "holdout"}:
                return {key: list(items) for key, items in value.items()}
    raise RuntimeError(f"Could not find exp03 split in {path}")


exp03_split = read_exp03_split(EXP03_CONFIG_PATH)
exp03_train_ids = exp03_split["train"]
exp03_holdout_ids = exp03_split["holdout"]
if len(exp03_train_ids) != 183 or len(exp03_holdout_ids) != 12:
    raise RuntimeError("Unexpected exp03 183/12 split")
if set(exp03_train_ids) & set(exp03_holdout_ids):
    raise RuntimeError("exp03 train and holdout overlap")

all_ids = sorted(path.stem for path in DATA_DIR.glob("*.geff"))
if len(all_ids) != 199 or len(set(all_ids)) != 199:
    raise RuntimeError(f"Expected 199 source videos, found {len(all_ids)}")
missing_exp03_ids = sorted(set(exp03_train_ids + exp03_holdout_ids) - set(all_ids))
if missing_exp03_ids:
    raise RuntimeError(f"exp03 split contains unknown videos: {missing_exp03_ids}")
missing_pairs = [
    str(DATA_DIR / f"{dataset_id}{suffix}")
    for dataset_id in all_ids
    for suffix in (".geff", ".zarr")
    if not (DATA_DIR / f"{dataset_id}{suffix}").exists()
]
if missing_pairs:
    raise FileNotFoundError("Source data are incomplete:\n" + "\n".join(missing_pairs[:20]))

RUN_DIR.mkdir(parents=True, exist_ok=True)
PREDICTIONS_ROOT.mkdir(parents=True, exist_ok=True)
EVALUATION_ROOT.mkdir(parents=True, exist_ok=True)
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
atomic_write_text(LOEO_TRAIN_SPLIT_PATH, json.dumps([{"split": 0, "train": train_ids, "test": holdout_ids}], indent=2))
atomic_write_text(LOEO_EVAL_SPLIT_PATH, json.dumps([{"split": 0, "train": [], "test": holdout_ids}], indent=2))
atomic_write_text(EXP03_TRAIN_SPLIT_PATH, json.dumps([{"split": 0, "train": exp03_train_ids, "test": exp03_holdout_ids}], indent=2))
atomic_write_text(EXP03_EVAL_SPLIT_PATH, json.dumps([{"split": 0, "train": [], "test": exp03_holdout_ids}], indent=2))
atomic_write_text(ALL_MANIFEST_PATH, json.dumps({"all": all_ids}, indent=2))

branch = checked([*GIT, "branch", "--show-current"], cwd=PROJECT_ROOT).strip()
if AUTO_GIT and branch != EXPECTED_BRANCH:
    raise RuntimeError(f"Auto-publish needs branch {EXPECTED_BRANCH!r}; current branch is {branch!r}")
if RUN_PIPELINE:
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is unavailable. Select the project Python 3.12 kernel and restart Jupyter.")
    probe = torch.zeros(1, device="cuda")
    del probe
    torch.cuda.empty_cache()
    cuda_device = torch.cuda.get_device_name(0)
    print(f"CUDA OK: torch={torch.__version__}, runtime={torch.version.cuda}, device={cuda_device}")
else:
    cuda_device = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "dry preflight"


def prepare_repo_copy() -> None:
    marker = REPO_DIR / ".friend_a_dense_ft_patch_v1"
    if marker.is_file():
        return
    if REPO_DIR.exists():
        guarded_remove_tree(REPO_DIR, RUN_DIR)
    REPO_DIR.mkdir(parents=True)
    for subdir in ("scripts", "src"):
        shutil.copytree(PROJECT_ROOT / "tools" / "repo" / subdir, REPO_DIR / subdir)
    train_patch = import_file("friend_a_dense_ft_train_patches", TRAIN_PATCH_PATH)
    predict_patch = import_file("friend_a_dense_ft_predict_patches", PREDICT_PATCH_PATH)
    train_script = REPO_DIR / "scripts" / "train_unet_transformer.py"
    predict_script = REPO_DIR / "scripts" / "predict_unet_transformer.py"
    train_patch.apply_train_patches(train_script)
    predict_patch.apply_predict_patches(predict_script)
    legacy_import = "from biohub_tracking.models import SimpleNodeTransformer, TemporalUNet3D"
    trainer_source = train_script.read_text(encoding="utf-8")
    if legacy_import in trainer_source:
        trainer_source = trainer_source.replace(
            legacy_import,
            "from biohub_tracking.models.simple_node_transformer import SimpleNodeTransformer\n"
            "from biohub_tracking.models.temporal_unet import TemporalUNet3D",
            1,
        )
        train_script.write_text(trainer_source, encoding="utf-8")
    compile(train_script.read_text(encoding="utf-8"), str(train_script), "exec")
    compile(predict_script.read_text(encoding="utf-8"), str(predict_script), "exec")
    help_text = checked([sys.executable, str(train_script), "--help"], cwd=REPO_DIR, env=common_env(DATA_DIR))
    for flag in ("--init-weights", "--max-iters", "--amp", "--no-grad-ckpt", "--max-detections", "--aug-intensity"):
        if flag not in help_text:
            raise RuntimeError(f"Patched trainer is missing {flag}")
    atomic_write_text(marker, "friend_a_dense_ft_patch_v1\n")


def download_model_file(dataset: str, remote: str, destination: Path) -> None:
    download_dir = destination.parent / "download"
    download_dir.mkdir(parents=True, exist_ok=True)
    output = checked(
        [sys.executable, "-m", "kaggle", "datasets", "download", dataset, "-f", remote,
         "-p", str(download_dir), "--unzip", "-o"],
        cwd=PROJECT_ROOT,
        env={**os.environ, "PYTHONUTF8": "1", "PYTHONIOENCODING": "utf-8"},
    )
    if output.strip():
        print(output.strip())
    matches = [path for path in download_dir.rglob(Path(remote).name) if path.is_file()]
    if not matches:
        raise FileNotFoundError(f"Kaggle download did not produce {remote}")
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(matches[0], destination)


def copy_verified_pair(source_dir: Path, target_dir: Path, expected_hash: str) -> Path:
    source_weights = source_dir / "edge_predictor_best.pth"
    source_config = source_dir / "config.json"
    if not source_weights.is_file() or not source_config.is_file():
        raise FileNotFoundError(f"Incomplete checkpoint pair in {source_dir}")
    if sha256_file(source_weights).lower() != expected_hash.lower():
        raise RuntimeError(f"Checkpoint hash mismatch in {source_dir}: {sha256_file(source_weights)}")
    target_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source_weights, target_dir / source_weights.name)
    shutil.copy2(source_config, target_dir / source_config.name)
    return target_dir / source_weights.name


def ensure_pair(target_dir: Path, expected_hash: str, local_candidates: list[Path], dataset: str, remote_folder: str) -> Path:
    target = target_dir / "edge_predictor_best.pth"
    config = target_dir / "config.json"
    if target.is_file() and config.is_file() and sha256_file(target).lower() == expected_hash.lower():
        return target
    source = next((path for path in local_candidates if (path / "edge_predictor_best.pth").is_file()), None)
    if source is not None:
        return copy_verified_pair(source, target_dir, expected_hash)
    for name in ("edge_predictor_best.pth", "config.json"):
        download_model_file(dataset, f"{remote_folder}/{name}", target_dir / name)
    if sha256_file(target).lower() != expected_hash.lower():
        raise RuntimeError(f"Downloaded checkpoint hash mismatch: {target}")
    return target


def ensure_primary_model() -> Path:
    return ensure_pair(
        PRIMARY_WEIGHTS_DIR,
        PRIMARY_EXPECTED_SHA256,
        [OUTPUT_ROOT / "_reference_weights" / "reference_pilkwang_primary"],
        PRIMARY_DATASET,
        PRIMARY_REMOTE_FOLDER,
    )


def ensure_teacher_models() -> tuple[Path, Path]:
    primary = ensure_pair(
        TEACHER_PRIMARY_DIR,
        TEACHER_PRIMARY_EXPECTED_SHA256,
        [
            OUTPUT_ROOT / "_dense_pipeline_weights" / "exp03B",
            OUTPUT_ROOT / "kaggle_weights_upload_20260907_2215" / "exp03B",
        ],
        WEIGHTS_DATASET,
        "exp03B",
    )
    secondary = ensure_pair(
        TEACHER_SECONDARY_DIR,
        TEACHER_SECONDARY_EXPECTED_SHA256,
        [
            OUTPUT_ROOT / "_dense_pipeline_weights" / "seed314159",
            OUTPUT_ROOT / "_reference_weights" / "reference_pilkwang_secondary_seed314159",
        ],
        SECONDARY_DATASET,
        SECONDARY_REMOTE_FOLDER,
    )
    return primary, secondary


prepare_repo_copy()
run_config = {
    "experiment": EXPERIMENT,
    "plan_revision": "2026-09-10 20:00",
    "branch": branch,
    "python": sys.version,
    "platform": platform.platform(),
    "cuda_device": cuda_device,
    "loeo_manifest_sha256": sha256_file(MANIFEST_PATH),
    "initialization": "verified pilkwang primary",
    "primary_sha256": PRIMARY_EXPECTED_SHA256,
    "variants": VARIANTS,
    "batch_size": BATCH_SIZE,
    "epochs": FT_EPOCHS,
    "learning_rate": FT_LEARNING_RATE,
    "learning_rate_schedule": "constant",
    "max_detections": MAX_DETECTIONS,
    "seed": SEED,
    "winning_delta": WINNING_DELTA,
    "scratch_training": False,
    "leaderboard_submission": False,
}
fingerprint = hashlib.sha256(json.dumps(run_config, sort_keys=True).encode()).hexdigest()
run_config["fingerprint"] = fingerprint
run_config_path = RUN_DIR / "run_config.json"
if run_config_path.is_file():
    previous = json.loads(run_config_path.read_text(encoding="utf-8"))
    if previous.get("fingerprint") != fingerprint:
        raise RuntimeError("The active dense-FT run uses another recipe. Set BIOHUB_DENSE_FT_RUN_ID to a new value.")
else:
    atomic_write_text(run_config_path, json.dumps(run_config, ensure_ascii=False, indent=2))
save_state(status="prepared", config_fingerprint=fingerprint)
print(f"Preflight OK: LOEO {len(train_ids)}/{len(holdout_ids)}; exp03 {len(exp03_train_ids)}/{len(exp03_holdout_ids)}")
print(f"Reusable teacher files: {sum((TEACHER_DIR / f'{value}.geff').is_dir() for value in train_ids + holdout_ids)}/138")
if not RUN_PIPELINE:
    print("Dry preflight only: no teacher, training, upload, or git operation was started.")


In [ ]:
# Shared resumable helpers for STEP 1c, five fine-tune comparisons, and the winning 183-video run.

from geff import GeffMetadata

TRAIN_SCRIPT = REPO_DIR / "scripts" / "train_unet_transformer.py"
PREDICT_SCRIPT = REPO_DIR / "scripts" / "predict_unet_transformer.py"


def kaggle_files(dataset: str = WEIGHTS_DATASET) -> str:
    return checked(
        [sys.executable, "-m", "kaggle", "datasets", "files", dataset, "--page-size", "1000"],
        cwd=PROJECT_ROOT,
        env={**os.environ, "PYTHONUTF8": "1", "PYTHONIOENCODING": "utf-8"},
    )


def download_dataset_snapshot() -> Path:
    snapshot = RUN_DIR / "weights_dataset_snapshot"
    marker = snapshot / ".download_complete"
    if marker.is_file():
        return snapshot
    if snapshot.exists():
        guarded_remove_tree(snapshot, RUN_DIR)
    snapshot.mkdir(parents=True)
    stream_command(
        [sys.executable, "-m", "kaggle", "datasets", "download", WEIGHTS_DATASET,
         "-p", str(snapshot), "--unzip", "-o"],
        cwd=PROJECT_ROOT,
        env={**os.environ, "PYTHONUTF8": "1", "PYTHONIOENCODING": "utf-8"},
        log_path=RUN_DIR / "dataset_download.log",
        label="download_weights_dataset",
    )
    atomic_write_text(marker, "complete\n")
    return snapshot


def restore_geff_folder(snapshot: Path, folder: str, destination: Path, expected_ids: list[str]) -> None:
    destination.mkdir(parents=True, exist_ok=True)
    for index, dataset_id in enumerate(expected_ids, start=1):
        target = destination / f"{dataset_id}.geff"
        if target.is_dir():
            continue
        unpacked = [
            path for path in snapshot.rglob(f"{dataset_id}.geff")
            if path.is_dir() and folder in path.parts
        ]
        if unpacked:
            shutil.copytree(unpacked[0], target)
        else:
            archives = [
                path for path in snapshot.rglob(f"{dataset_id}.geff.zip")
                if folder in path.parts
            ]
            if not archives:
                raise FileNotFoundError(f"Remote folder {folder} has no {dataset_id}.geff(.zip)")
            temporary = RUN_DIR / "restore_tmp" / folder / dataset_id
            if temporary.exists():
                guarded_remove_tree(temporary, RUN_DIR)
            temporary.mkdir(parents=True)
            shutil.unpack_archive(str(archives[0]), str(temporary))
            nested = [path for path in temporary.rglob(f"{dataset_id}.geff") if path.is_dir()]
            shutil.copytree(nested[0] if nested else temporary, target)
        if index % 20 == 0 or index == len(expected_ids):
            print(f"restored {folder}: {index}/{len(expected_ids)}", flush=True)


def upsert_results_row(key: str, line: str) -> None:
    path = PROJECT_ROOT / "experiments" / "RESULTS.md"
    lines = [value for value in path.read_text(encoding="utf-8").splitlines() if not value.startswith(key)]
    insert_at = next((index for index, value in enumerate(lines) if value.startswith("## Per-video")), len(lines))
    lines.insert(insert_at, line)
    atomic_write_text(path, "\n".join(lines).rstrip() + "\n")


def commit_and_push(paths: list[Path], message: str) -> None:
    relative = [str(path.resolve().relative_to(PROJECT_ROOT.resolve())).replace("\\", "/") for path in paths]
    staged_before = checked([*GIT, "diff", "--cached", "--name-only"], cwd=PROJECT_ROOT).splitlines()
    unrelated = set(staged_before) - set(relative)
    if unrelated:
        raise RuntimeError("Refusing to include unrelated staged files: " + ", ".join(sorted(unrelated)))
    checked([*GIT, "add", "--", *relative], cwd=PROJECT_ROOT)
    staged = subprocess.run([*GIT, "diff", "--cached", "--quiet"], cwd=PROJECT_ROOT)
    if staged.returncode == 1:
        print(checked([*GIT, "commit", "-m", message], cwd=PROJECT_ROOT).strip())
    elif staged.returncode != 0:
        raise RuntimeError("git diff --cached failed")
    print(checked([*GIT, "push", "origin", EXPECTED_BRANCH], cwd=PROJECT_ROOT).strip())


def stage_current_dataset(name: str) -> Path:
    stage = RUN_DIR / name
    if stage.exists():
        guarded_remove_tree(stage, RUN_DIR)
    stage.mkdir(parents=True)
    stream_command(
        [sys.executable, "-m", "kaggle", "datasets", "download", WEIGHTS_DATASET,
         "-p", str(stage), "--unzip", "-o"],
        cwd=PROJECT_ROOT,
        env={**os.environ, "PYTHONUTF8": "1", "PYTHONIOENCODING": "utf-8"},
        log_path=RUN_DIR / f"{name}.log",
        label=f"stage_{name}",
    )
    atomic_write_text(
        stage / "dataset-metadata.json",
        json.dumps({"id": WEIGHTS_DATASET, "title": "biohub-ft-weights"}, indent=2) + "\n",
    )
    return stage


def publish_stage(stage: Path, message: str, expected: list[str]) -> None:
    stream_command(
        [sys.executable, "-m", "kaggle", "datasets", "version", "-p", str(stage),
         "--dir-mode", "zip", "-m", message],
        cwd=PROJECT_ROOT,
        env={**os.environ, "PYTHONUTF8": "1", "PYTHONIOENCODING": "utf-8"},
        log_path=RUN_DIR / "dataset_upload.log",
        label="upload_dataset_version",
    )
    for attempt in range(12):
        listing = kaggle_files()
        if all(value in listing for value in expected):
            return
        if attempt < 11:
            print("Kaggle is processing the version; retrying in 10 seconds...")
            time.sleep(10)
    raise RuntimeError("Uploaded dataset version is not yet visible with the expected files")


def upload_geff_folders(ids: list[str]) -> None:
    stage = stage_current_dataset("geff_upload_stage")
    for folder, source in ((TEACHER_DATASET_FOLDER, TEACHER_DIR), (DENSE_DATASET_FOLDER, DENSE_DATA_DIR)):
        destination = stage / folder
        if destination.exists():
            guarded_remove_tree(destination, stage)
        destination.mkdir(parents=True)
        for dataset_id in ids:
            geff = source / f"{dataset_id}.geff"
            if not geff.is_dir():
                raise FileNotFoundError(geff)
            shutil.copytree(geff, destination / geff.name)
    publish_stage(
        stage,
        f"Friend A dense teacher/labels: {len(ids)} videos",
        [f"{TEACHER_DATASET_FOLDER}/", f"{DENSE_DATASET_FOLDER}/"],
    )


def upload_weight_pair(weights: Path, folder: str, message: str) -> None:
    config = weights.with_name("config.json")
    if not weights.is_file() or not config.is_file():
        raise FileNotFoundError(f"Incomplete checkpoint pair: {weights}")
    digest = sha256_file(weights)
    if load_state().get(f"uploaded_{folder}") == digest:
        return
    stage = stage_current_dataset("winner_upload_stage")
    destination = stage / folder
    if destination.exists():
        guarded_remove_tree(destination, stage)
    destination.mkdir(parents=True)
    shutil.copy2(weights, destination / "edge_predictor_best.pth")
    shutil.copy2(config, destination / "config.json")
    publish_stage(stage, message, [f"{folder}/edge_predictor_best.pth", f"{folder}/config.json"])
    save_state(**{f"uploaded_{folder}": digest})


def prediction_output(user: str, method: str) -> Path:
    return PREDICTIONS_ROOT / user / method / "split_0"


def predict_teacher_chunk(ids: list[str], label: str) -> list[Path]:
    primary, secondary = ensure_teacher_models()
    split = RUN_DIR / "teacher_splits" / f"{label}.json"
    atomic_write_text(split, json.dumps([{"split": 0, "train": [], "test": ids}], indent=2))
    user = "friend_a_dense_teacher"
    env = common_env(DATA_DIR)
    env["USER"] = user
    env["BIOHUB_ENSEMBLE_WEIGHTS"] = str(secondary)
    stream_command(
        [sys.executable, str(PREDICT_SCRIPT), "--method", label, "--data-dir", str(DATA_DIR),
         "--splits", str(split), "--split", "0", "--weights", str(primary),
         "--unet-batch-size", "4", "--det-threshold", str(DET_THRESHOLD)],
        cwd=REPO_DIR,
        env=env,
        log_path=RUN_DIR / "teacher_logs" / f"{label}.log",
        label=label,
    )
    output = prediction_output(user, label)
    geffs = sorted(output.glob("*.geff"))
    if {path.stem for path in geffs} != set(ids):
        raise RuntimeError(f"Teacher coverage mismatch for {label}")
    return geffs


def ensure_teacher(ids: list[str], *, smoke_key: str) -> None:
    TEACHER_DIR.mkdir(parents=True, exist_ok=True)
    missing = [dataset_id for dataset_id in ids if not (TEACHER_DIR / f"{dataset_id}.geff").is_dir()]
    if missing and not load_state().get(smoke_key):
        smoke = predict_teacher_chunk(missing[:2], f"{smoke_key}_2")
        if len(smoke) != min(2, len(missing)):
            raise RuntimeError("Teacher smoke failed")
        guarded_remove_tree(prediction_output("friend_a_dense_teacher", f"{smoke_key}_2"), PREDICTIONS_ROOT)
        save_state(**{smoke_key: True})
    missing = [dataset_id for dataset_id in ids if not (TEACHER_DIR / f"{dataset_id}.geff").is_dir()]
    for offset in range(0, len(missing), TEACHER_CHUNK_SIZE):
        chunk = missing[offset: offset + TEACHER_CHUNK_SIZE]
        label = f"teacher_{smoke_key}_{offset // TEACHER_CHUNK_SIZE:03d}"
        geffs = predict_teacher_chunk(chunk, label)
        for source in geffs:
            destination = TEACHER_DIR / source.name
            if destination.exists():
                guarded_remove_tree(destination, TEACHER_DIR)
            shutil.move(str(source), str(destination))
        output = prediction_output("friend_a_dense_teacher", label)
        if output.exists():
            guarded_remove_tree(output, PREDICTIONS_ROOT)
        save_state(status="teacher_in_progress", teacher_complete_count=sum((TEACHER_DIR / f"{x}.geff").is_dir() for x in ids))
    if any(not (TEACHER_DIR / f"{dataset_id}.geff").is_dir() for dataset_id in ids):
        raise RuntimeError("Teacher generation ended with missing GEFFs")


def build_dense(
    state_label: str,
    manifest_path: Path,
    manifest_keys: str,
    ids: list[str],
    stats_path: Path,
    *,
    smoke: bool,
) -> None:
    if smoke and not load_state().get(f"{state_label}_builder_smoke"):
        smoke_dir = RUN_DIR / f"{state_label}_builder_smoke"
        stream_command(
            [sys.executable, str(BUILDER_PATH), "--data-dir", str(DATA_DIR), "--teacher-dir", str(TEACHER_DIR),
             "--output-dir", str(smoke_dir), "--manifest", str(manifest_path), "--manifest-keys", manifest_keys,
             "--dedup-radius-um", str(TEACHER_DEDUP_RADIUS_UM), "--stats-out", str(smoke_dir / "dense_stats.csv"),
             "--limit", "2"],
            cwd=PROJECT_ROOT,
            env=common_env(DATA_DIR),
            log_path=RUN_DIR / f"{state_label}_builder_smoke.log",
            label=f"{state_label}_builder_smoke",
        )
        save_state(**{f"{state_label}_builder_smoke": True})
    stream_command(
        [sys.executable, str(BUILDER_PATH), "--data-dir", str(DATA_DIR), "--teacher-dir", str(TEACHER_DIR),
         "--output-dir", str(DENSE_DATA_DIR), "--manifest", str(manifest_path), "--manifest-keys", manifest_keys,
         "--dedup-radius-um", str(TEACHER_DEDUP_RADIUS_UM), "--stats-out", str(stats_path)],
        cwd=PROJECT_ROOT,
        env=common_env(DATA_DIR),
        log_path=RUN_DIR / f"build_dense_{state_label}.log",
        label=f"build_dense_{state_label}",
    )
    with stats_path.open(newline="", encoding="utf-8") as handle:
        rows = list(csv.DictReader(handle))
    if len(rows) != len(ids) or any(float(row["gt_node_recall_in_dense"]) != 1.0 for row in rows):
        raise RuntimeError(f"Dense stats validation failed for {state_label}")


def finetune_command(
    *, method: str, data_dir: Path, split_path: Path, max_iters: int,
    det_neg_weight: float, aug_intensity: bool,
) -> list[str]:
    primary = ensure_primary_model()
    command = [
        sys.executable, str(TRAIN_SCRIPT), "--method", method, "--data-dir", str(data_dir),
        "--splits", str(split_path), "--split", "0", "--epochs", str(FT_EPOCHS),
        "--max-iters", str(max_iters), "--batch-size", str(BATCH_SIZE), "--num-workers", str(NUM_WORKERS),
        "--lr", str(FT_LEARNING_RATE), "--lr-schedule", "constant", "--det-loss", "bce",
        "--det-loss-weight", str(DET_LOSS_WEIGHT), "--det-neg-weight", str(det_neg_weight),
        "--window-size", str(WINDOW_SIZE), "--downsample", DOWNSAMPLE,
        "--pool-kernel-um", str(POOL_KERNEL_UM), "--max-detections", str(MAX_DETECTIONS),
        "--seed", str(SEED), "--amp", "--no-grad-ckpt", "--single-gpu",
        "--init-weights", str(primary),
    ]
    if aug_intensity:
        command.append("--aug-intensity")
    if command.count("--init-weights") != 1 or "--unet-weights" in command:
        raise RuntimeError("Every revised-plan training command must fine-tune exactly one full primary checkpoint")
    return command


def training_output(method: str) -> Path:
    return REPO_DIR / "weights" / method / "split_0"


def smoke_finetune(label: str, data_dir: Path, det_neg_weight: float, aug_intensity: bool) -> None:
    key = f"smoke_{label}"
    if load_state().get(key):
        return
    split = RUN_DIR / "smoke_splits" / f"{label}.json"
    atomic_write_text(split, json.dumps([{"split": 0, "train": train_ids[:2], "test": train_ids[:2]}], indent=2))
    method = f"smoke_{label}"
    output = training_output(method)
    if output.exists():
        guarded_remove_tree(output, REPO_DIR / "weights")
    command = finetune_command(
        method=method, data_dir=data_dir, split_path=split, max_iters=2,
        det_neg_weight=det_neg_weight, aug_intensity=aug_intensity,
    )
    stream_command(command, cwd=REPO_DIR, env=common_env(data_dir), log_path=RUN_DIR / f"smoke_{label}.log", label=f"smoke_{label}")
    if not (output / "edge_predictor_best.pth").is_file():
        raise RuntimeError(f"Fine-tune smoke produced no weights: {label}")
    save_state(**{key: True})


def run_finetune(
    *, label: str, data_dir: Path, split_path: Path, max_iters: int,
    det_neg_weight: float, aug_intensity: bool,
) -> Path:
    state = load_state()
    completed = state.get("completed_trainings", {})
    recorded = completed.get(label)
    if recorded and Path(recorded["weights"]).is_file():
        print(f"Skipping completed fine-tune: {label}")
        return Path(recorded["weights"]).parent
    smoke_finetune(label, data_dir, det_neg_weight, aug_intensity)
    method = f"friend_a_{label}"
    output = training_output(method)
    if output.exists():
        guarded_remove_tree(output, REPO_DIR / "weights")
    command = finetune_command(
        method=method, data_dir=data_dir, split_path=split_path, max_iters=max_iters,
        det_neg_weight=det_neg_weight, aug_intensity=aug_intensity,
    )
    atomic_write_text(RUN_DIR / f"{label}_command.txt", subprocess.list2cmdline(command) + "\n")
    save_state(status=f"training_{label}", active_training=label)
    stream_command(command, cwd=REPO_DIR, env=common_env(data_dir), log_path=RUN_DIR / f"train_{label}.log", label=f"train_{label}")
    weights = output / "edge_predictor_best.pth"
    if not weights.is_file() or not weights.with_name("config.json").is_file():
        raise FileNotFoundError(f"Fine-tune did not produce a complete checkpoint: {output}")
    completed = dict(load_state().get("completed_trainings", {}))
    completed[label] = {"weights": str(weights), "sha256": sha256_file(weights), "max_iters": max_iters}
    save_state(status=f"training_{label}_complete", active_training=None, completed_trainings=completed)
    return output


def metric_node_counts(rows: list[dict]) -> tuple[int, float, float]:
    predicted = 0
    estimated = 0.0
    for row in rows:
        predicted += int(row["num_pred_nodes"])
        metadata = GeffMetadata.read(DATA_DIR / f"{row['dataset']}.geff")
        value = float((metadata.extra or {}).get("estimated_number_of_nodes", float("nan")))
        if not math.isfinite(value) or value <= 0:
            raise RuntimeError(f"Missing estimated_number_of_nodes for {row['dataset']}")
        estimated += value
    return predicted, estimated, predicted / estimated


def evaluate_weights(label: str, weights: Path, eval_split: Path, expected_ids: list[str]) -> dict:
    evaluation = EVALUATION_ROOT / label
    metrics_path = evaluation / "official_metrics.json"
    if metrics_path.is_file():
        payload = json.loads(metrics_path.read_text(encoding="utf-8"))
        archive = payload.get("prediction_zip")
        if int(payload.get("summary", {}).get("n", -1)) == len(expected_ids) and archive and Path(archive).is_file():
            print(f"Skipping completed official evaluation: {label}")
            return payload
    evaluation.mkdir(parents=True, exist_ok=True)
    user = "friend_a_dense_ft_eval"
    env = common_env(DATA_DIR)
    env["USER"] = user
    prediction_dir = prediction_output(user, label)
    if prediction_dir.exists():
        guarded_remove_tree(prediction_dir, PREDICTIONS_ROOT)
    stream_command(
        [sys.executable, str(PREDICT_SCRIPT), "--method", label, "--data-dir", str(DATA_DIR),
         "--splits", str(eval_split), "--split", "0", "--weights", str(weights),
         "--unet-batch-size", "4", "--det-threshold", str(DET_THRESHOLD), "--use-ilp",
         "--ilp-appearance-weight", str(ILP_APPEARANCE), "--ilp-disappearance-weight", str(ILP_DISAPPEARANCE),
         "--ilp-division-weight", str(ILP_DIVISION)],
        cwd=REPO_DIR,
        env=env,
        log_path=evaluation / "predict.log",
        label=f"predict_{label}",
    )
    geffs = sorted(prediction_dir.glob("*.geff"))
    if {path.stem for path in geffs} != set(expected_ids):
        raise RuntimeError(f"Prediction coverage mismatch for {label}")
    stream_command(
        [sys.executable, str(PROJECT_ROOT / "tools" / "score_geffs.py"), str(prediction_dir), "--out", str(metrics_path)],
        cwd=PROJECT_ROOT,
        env={**os.environ, "BIOHUB_DATA_DIR": str(DATA_DIR), "PYTHONUTF8": "1", "PYTHONIOENCODING": "utf-8"},
        log_path=evaluation / "score.log",
        label=f"score_{label}",
    )
    payload = json.loads(metrics_path.read_text(encoding="utf-8"))
    if int(payload.get("summary", {}).get("n", -1)) != len(expected_ids):
        raise RuntimeError(f"Official scorer skipped videos for {label}")
    n_pred, n_est, ratio = metric_node_counts(payload["rows"])
    zip_dir = RUN_DIR / "prediction_zips"
    zip_dir.mkdir(parents=True, exist_ok=True)
    temporary = Path(shutil.make_archive(str(zip_dir / f".{label}.building"), "zip", root_dir=prediction_dir))
    archive = zip_dir / f"{label}.zip"
    temporary.replace(archive)
    guarded_remove_tree(prediction_dir, PREDICTIONS_ROOT)
    payload["checkpoint"] = {"weights": str(weights), "weights_sha256": sha256_file(weights)}
    payload["aggregate_node_counts"] = {"n_pred": n_pred, "n_est": n_est, "n_pred_over_n_est": ratio}
    payload["prediction_zip"] = str(archive)
    atomic_write_text(metrics_path, json.dumps(payload, ensure_ascii=False, indent=2))
    print(
        f"{label}: score={payload['summary']['score']:.4f}, edge={payload['summary']['edge_jaccard']:.4f}, "
        f"recall={payload['summary']['node_recall']:.4f}, Npred/Nest={ratio:.3f}"
    )
    return payload


def compact_metric(payload: dict) -> dict:
    summary = payload["summary"]
    counts = payload["aggregate_node_counts"]
    return {
        "score": float(summary["score"]),
        "adj_edge_jaccard": float(summary["adj_edge_jaccard"]),
        "edge_jaccard": float(summary["edge_jaccard"]),
        "division_jaccard": float(summary["division_jaccard"]),
        "division_tp": int(summary["division_tp"]),
        "division_fp": int(summary["division_fp"]),
        "division_fn": int(summary["division_fn"]),
        "node_recall": float(summary["node_recall"]),
        "n_pred": int(counts["n_pred"]),
        "n_est": float(counts["n_est"]),
        "n_pred_over_n_est": float(counts["n_pred_over_n_est"]),
        "weights": payload["checkpoint"]["weights"],
        "weights_sha256": payload["checkpoint"]["weights_sha256"],
    }


def record_variant(variant: dict, metric: dict) -> dict:
    results_path = EXPERIMENT_DIR / "results.json"
    result = json.loads(results_path.read_text(encoding="utf-8")) if results_path.is_file() else {
        "experiment": EXPERIMENT,
        "plan_revision": "2026-09-10 20:00",
        "split": "126x6bba fine-tune -> 12x44b6 raw official validation",
        "initialization": "pilkwang primary",
        "primary_sha256": PRIMARY_EXPECTED_SHA256,
        "variants": {},
        "winning_delta": WINNING_DELTA,
        "leaderboard_submission": False,
    }
    result["variants"][variant["name"]] = {**variant, "metric": metric, "completed_at": datetime.now().astimezone().isoformat()}
    baseline = result["variants"].get("sparse_1200", {}).get("metric", {}).get("score")
    if baseline is not None:
        for name, value in result["variants"].items():
            value["delta_vs_sparse"] = value["metric"]["score"] - baseline
    result["updated_at"] = datetime.now().astimezone().isoformat()
    atomic_write_text(results_path, json.dumps(result, ensure_ascii=False, indent=2))
    label = variant["name"]
    upsert_results_row(
        f"| local Friend A exp03 {label} |",
        f"| local Friend A exp03 {label} | primary FT; {variant['data']} labels; {variant['max_iters']} it; "
        f"lr 5e-5 constant; neg {variant['det_neg_weight']}; aug={variant['aug_intensity']} | - | "
        f"**{metric['score']:.4f}** (edge {metric['edge_jaccard']:.4f}, recall {metric['node_recall']:.4f}, "
        f"Npred/Nest {metric['n_pred_over_n_est']:.3f}) LOEO-44b6 | GPU | init pilkwang primary; no LB |",
    )
    completed_summary = ", ".join(
        f"{name}={value['metric']['score']:.4f}" for name, value in result["variants"].items()
    )
    context_path = PROJECT_ROOT / "CONTEXT.md"
    update_marked_block(
        context_path,
        "<!-- friend-a-exp03-dense-ft-primary-progress -->",
        f"- {datetime.now().date().isoformat()} Friend A primary fine-tune comparison in progress/completed: "
        f"{completed_summary}; raw official LOEO-44b6, gate dense >= sparse +0.005; no LB.",
        "## 14. Operational state",
    )
    if AUTO_GIT:
        commit_and_push(
            [results_path, PROJECT_ROOT / "experiments" / "RESULTS.md", context_path],
            f"Record Friend A primary FT {label}",
        )
    return result


In [ ]:
# STEP 1c: finish/reuse the 138-video teacher and dense labels before any fine-tuning.

step1c_ids = train_ids + holdout_ids

if RUN_PIPELINE and not load_state().get("step1c_complete"):
    listing = kaggle_files()
    remote_teacher = f"{TEACHER_DATASET_FOLDER}/" in listing
    remote_dense = f"{DENSE_DATASET_FOLDER}/" in listing
    print(f"Kaggle dataset: teacher={remote_teacher}, dense={remote_dense}")

    if remote_teacher or remote_dense:
        snapshot = download_dataset_snapshot()
        if remote_teacher:
            restore_geff_folder(snapshot, TEACHER_DATASET_FOLDER, TEACHER_DIR, step1c_ids)
        if remote_dense:
            restore_geff_folder(snapshot, DENSE_DATASET_FOLDER, DENSE_DATA_DIR, step1c_ids)

    ensure_teacher(step1c_ids, smoke_key="step1c_teacher_smoke")
    tracked_stats = EXPERIMENT_DIR / "dense_stats_138.csv"
    build_dense(
        "step1c_138",
        MANIFEST_PATH,
        "train,holdout",
        step1c_ids,
        tracked_stats,
        smoke=True,
    )

    if AUTO_UPLOAD and not (remote_teacher and remote_dense):
        upload_geff_folders(step1c_ids)
    elif not AUTO_UPLOAD:
        print("BIOHUB_DENSE_FT_AUTO_UPLOAD=0: teacher/dense upload skipped by request.")

    with tracked_stats.open(newline="", encoding="utf-8") as handle:
        dense_rows = list(csv.DictReader(handle))
    ratios = [float(row["n_dense_over_estimated"]) for row in dense_rows]
    recalls = [float(row["teacher_recall_of_gt_5um"]) for row in dense_rows]
    result = {
        "experiment": EXPERIMENT,
        "step": "1c",
        "status": "complete",
        "completed_at": datetime.now().astimezone().isoformat(),
        "videos": len(step1c_ids),
        "train_6bba": len(train_ids),
        "diagnostic_44b6": len(holdout_ids),
        "teacher": "exp03B + pilkwang seed314159 logits ensemble",
        "teacher_det_threshold": DET_THRESHOLD,
        "teacher_use_ilp": False,
        "dense_rule": "teacher union GT; GT wins within 5 um; GT edges only",
        "teacher_recall_of_gt_5um_mean": sum(recalls) / len(recalls),
        "n_dense_over_estimated_mean": sum(ratios) / len(ratios),
        "n_dense_over_estimated_min": min(ratios),
        "n_dense_over_estimated_max": max(ratios),
        "ratio_outliers": sum(str(row["ratio_outside_0.9_1.1"]).lower() == "true" for row in dense_rows),
        "stats_csv": str(tracked_stats.relative_to(PROJECT_ROOT)).replace("\\", "/"),
        "dataset": WEIGHTS_DATASET,
        "dataset_folders": [TEACHER_DATASET_FOLDER, DENSE_DATASET_FOLDER],
        "uploaded": bool(AUTO_UPLOAD or (remote_teacher and remote_dense)),
        "leaderboard_submission": False,
    }
    step1c_result_path = EXPERIMENT_DIR / "step1c_results.json"
    atomic_write_text(step1c_result_path, json.dumps(result, ensure_ascii=False, indent=2))
    upsert_results_row(
        "| local Friend A exp03 STEP 1c |",
        f"| local Friend A exp03 STEP 1c | exp03B+seed314159 teacher (det {DET_THRESHOLD}, no ILP) -> dense GT union | - | "
        f"138 videos; teacher recall@5um mean {result['teacher_recall_of_gt_5um_mean']:.4f}; "
        f"dense/Nest mean {result['n_dense_over_estimated_mean']:.3f} | teacher GPU + CPU build | "
        f"GT recall 1.0; folders `{TEACHER_DATASET_FOLDER}/`, `{DENSE_DATASET_FOLDER}/`; no LB |",
    )
    results_md = PROJECT_ROOT / "experiments" / "RESULTS.md"
    context_path = PROJECT_ROOT / "CONTEXT.md"
    update_marked_block(
        context_path,
        "<!-- friend-a-step1c-dense-result -->",
        f"- {datetime.now().date().isoformat()} Friend A STEP 1c complete: exp03B+seed314159 teacher and dense labels "
        f"for 126 train 6bba + 12 diagnostic 44b6; teacher recall@5um mean "
        f"{result['teacher_recall_of_gt_5um_mean']:.4f}, dense/N_est mean "
        f"{result['n_dense_over_estimated_mean']:.3f}; dataset folders `{TEACHER_DATASET_FOLDER}/` and "
        f"`{DENSE_DATASET_FOLDER}/`; 44b6 pseudo-points are not used for training.",
        heading_prefix="## 14. Operational state",
    )
    save_state(status="step1c_complete", step1c_complete=True)
    if AUTO_GIT:
        commit_and_push(
            [tracked_stats, step1c_result_path, results_md, context_path],
            "Complete Friend A STEP 1c dense labels",
        )
    print("STEP 1c complete. No leaderboard submission was made.")
elif RUN_PIPELINE:
    print("Skipping completed STEP 1c.")
else:
    print("Dry preflight: STEP 1c was not started.")


In [ ]:
# STEP 2: same-machine sparse reference plus four dense primary fine-tunes.

if RUN_PIPELINE:
    if not load_state().get("step1c_complete"):
        raise RuntimeError("STEP 1c is incomplete; refusing to train before dense labels are verified.")

    ensure_primary_model()
    experiment_result = None
    for variant in VARIANTS:
        data_dir = DATA_DIR if variant["data"] == "sparse" else DENSE_DATA_DIR
        output = run_finetune(
            label=variant["name"],
            data_dir=data_dir,
            split_path=LOEO_TRAIN_SPLIT_PATH,
            max_iters=variant["max_iters"],
            det_neg_weight=variant["det_neg_weight"],
            aug_intensity=variant["aug_intensity"],
        )
        payload = evaluate_weights(
            variant["name"],
            output / "edge_predictor_best.pth",
            LOEO_EVAL_SPLIT_PATH,
            holdout_ids,
        )
        experiment_result = record_variant(variant, compact_metric(payload))

    assert experiment_result is not None
    sparse_score = experiment_result["variants"]["sparse_1200"]["metric"]["score"]
    dense_names = [variant["name"] for variant in VARIANTS if variant["data"] == "dense"]
    best_dense_name = max(dense_names, key=lambda name: experiment_result["variants"][name]["metric"]["score"])
    best_dense_score = experiment_result["variants"][best_dense_name]["metric"]["score"]
    delta = best_dense_score - sparse_score
    winner = best_dense_name if delta >= WINNING_DELTA else None
    experiment_result.update(
        {
            "status": "step2_complete",
            "sparse_reference_score": sparse_score,
            "best_dense_variant": best_dense_name,
            "best_dense_score": best_dense_score,
            "best_dense_delta_vs_sparse": delta,
            "winning_variant": winner,
            "step3_allowed": winner is not None,
            "completed_at": datetime.now().astimezone().isoformat(),
        }
    )
    results_path = EXPERIMENT_DIR / "results.json"
    atomic_write_text(results_path, json.dumps(experiment_result, ensure_ascii=False, indent=2))
    decision = (
        f"winner {winner}: {best_dense_score:.4f} vs sparse {sparse_score:.4f} ({delta:+.4f})"
        if winner
        else f"no dense winner: best {best_dense_name} {best_dense_score:.4f} vs sparse {sparse_score:.4f} ({delta:+.4f})"
    )
    context_path = PROJECT_ROOT / "CONTEXT.md"
    update_marked_block(
        context_path,
        "<!-- friend-a-exp03-dense-ft-primary-decision -->",
        f"- {datetime.now().date().isoformat()} Friend A primary dense-FT decision on raw LOEO-44b6: {decision}; "
        f"required delta +{WINNING_DELTA:.3f}; no LB.",
        heading_prefix="## 14. Operational state",
    )
    save_state(
        status="step2_complete",
        step2_complete=True,
        sparse_reference_score=sparse_score,
        best_dense_variant=best_dense_name,
        best_dense_score=best_dense_score,
        best_dense_delta_vs_sparse=delta,
        winning_variant=winner,
    )
    if AUTO_GIT:
        commit_and_push([results_path, context_path], "Record Friend A primary dense FT decision")
    print(f"STEP 2 complete: {decision}")
else:
    print("Dry preflight: STEP 2 fine-tunes were not started.")


In [ ]:
# STEP 3: only a winning dense recipe is extended to the fixed exp03 183-video training split.

if RUN_PIPELINE and RUN_STEP3:
    state = load_state()
    if not state.get("step2_complete"):
        raise RuntimeError("STEP 2 is incomplete; refusing to choose a recipe early.")
    winner_name = state.get("winning_variant")
    if winner_name is None:
        print("STEP 3 correctly skipped: no dense variant cleared the +0.005 gate.")
    elif state.get("step3_complete"):
        print("Skipping completed STEP 3.")
    else:
        winner = next(variant for variant in VARIANTS if variant["name"] == winner_name)
        print(f"STEP 3 winner: {winner_name}. Extending teacher/dense labels to all 199 videos.")
        ensure_teacher(all_ids, smoke_key="all199_teacher_smoke")
        tracked_stats = EXPERIMENT_DIR / "dense_stats_199.csv"
        build_dense(
            "all199",
            ALL_MANIFEST_PATH,
            "all",
            all_ids,
            tracked_stats,
            smoke=True,
        )
        if AUTO_UPLOAD:
            upload_geff_folders(all_ids)
        else:
            print("BIOHUB_DENSE_FT_AUTO_UPLOAD=0: updated 199-video GEFF upload skipped by request.")

        step3_label = f"step3_183_{winner_name}"
        output = run_finetune(
            label=step3_label,
            data_dir=DENSE_DATA_DIR,
            split_path=EXP03_TRAIN_SPLIT_PATH,
            max_iters=winner["max_iters"],
            det_neg_weight=winner["det_neg_weight"],
            aug_intensity=winner["aug_intensity"],
        )
        weights = output / "edge_predictor_best.pth"
        loeo_payload = evaluate_weights(
            f"{step3_label}_loeo44b6_mixed_overlap",
            weights,
            LOEO_EVAL_SPLIT_PATH,
            holdout_ids,
        )
        exp03_payload = evaluate_weights(
            f"{step3_label}_exp03_hold12",
            weights,
            EXP03_EVAL_SPLIT_PATH,
            exp03_holdout_ids,
        )
        loeo_metric = compact_metric(loeo_payload)
        exp03_metric = compact_metric(exp03_payload)
        if AUTO_UPLOAD:
            upload_weight_pair(
                weights,
                WINNER_WEIGHTS_FOLDER,
                f"Friend A primary dense fine-tune on exp03 183: {winner_name}",
            )
        else:
            print("BIOHUB_DENSE_FT_AUTO_UPLOAD=0: final weights upload skipped by request.")

        result = {
            "experiment": EXPERIMENT,
            "step": 3,
            "status": "complete",
            "completed_at": datetime.now().astimezone().isoformat(),
            "winning_recipe": winner,
            "train_split": "exp03 fixed 183 non-holdout videos",
            "initialization": "pilkwang primary",
            "weights": str(weights),
            "weights_sha256": sha256_file(weights),
            "dataset": WEIGHTS_DATASET,
            "dataset_folder": WINNER_WEIGHTS_FOLDER,
            "uploaded": AUTO_UPLOAD,
            "loeo_44b6_mixed_overlap_diagnostic": loeo_metric,
            "exp03_hold12_official": exp03_metric,
            "leaderboard_submission": False,
        }
        step3_result_path = EXPERIMENT_DIR / "step3_results.json"
        atomic_write_text(step3_result_path, json.dumps(result, ensure_ascii=False, indent=2))
        upsert_results_row(
            "| local Friend A exp03 STEP 3 |",
            f"| local Friend A exp03 STEP 3 | primary dense FT `{winner_name}` on fixed 183 | - | "
            f"hold12 **{exp03_metric['score']:.4f}** (edge {exp03_metric['edge_jaccard']:.4f}, "
            f"recall {exp03_metric['node_recall']:.4f}, Npred/Nest {exp03_metric['n_pred_over_n_est']:.3f}); "
            f"LOEO-44b6 mixed-overlap diagnostic {loeo_metric['score']:.4f} | GPU | "
            f"`{WINNER_WEIGHTS_FOLDER}/`; no LB |",
        )
        results_md = PROJECT_ROOT / "experiments" / "RESULTS.md"
        context_path = PROJECT_ROOT / "CONTEXT.md"
        update_marked_block(
            context_path,
            "<!-- friend-a-exp03-dense-ft-primary-step3 -->",
            f"- {datetime.now().date().isoformat()} Friend A STEP 3 complete: primary dense FT `{winner_name}` on fixed "
            f"183 videos; raw official exp03 hold12 {exp03_metric['score']:.4f}, LOEO-44b6 mixed-overlap diagnostic "
            f"{loeo_metric['score']:.4f}; weights `{WINNER_WEIGHTS_FOLDER}/`; no LB.",
            heading_prefix="## 14. Operational state",
        )
        save_state(status="step3_complete", step3_complete=True, step3_weights_sha256=sha256_file(weights))
        if AUTO_GIT:
            commit_and_push(
                [tracked_stats, step3_result_path, results_md, context_path],
                "Complete Friend A primary dense fine-tune",
            )
        print("\nMESSAGE FOR VITALII:")
        print(
            f"Friend A STEP 3 готов: `{winner_name}` дообучен от pilkwang primary на fixed exp03 183. "
            f"Raw official hold12={exp03_metric['score']:.4f}; веса и config.json: "
            f"{WEIGHTS_DATASET}/{WINNER_WEIGHTS_FOLDER}/. Можно пробовать как PRIMARY на TTA-base. "
            "На leaderboard я ничего не отправлял."
        )
elif RUN_PIPELINE:
    print("BIOHUB_DENSE_FT_RUN_STEP3=0: STEP 3 skipped by request.")
else:
    print("Dry preflight: STEP 3 was not started.")
